# Week 7: Deep Learning Foundations
## Notebook 14: Gradient Clipping: L2 Norm Clipping vs Value Clamping

### 1. Conceptual Intuition & Theoretical Background
In deep architectures and Recurrent Neural Networks (RNNs), consecutive matrix multiplications during backpropagation through time (BPTT) can cause gradients to explode exponentially:
$$\left\| \frac{\partial \mathcal{L}}{\partial W} \right\| \longrightarrow \infty$$
This leads to catastrophic weight corruption and `NaN` outputs.

**1. Value Clamping (Element-Wise):**
Clamps every individual partial derivative into range $[-\text{val}, \text{val}]$:
$$g_i = \max(-\text{clip\_val}, \min(\text{clip\_val}, g_i))$$
- **Flaw:** Distorts the gradient vector's **directional angle**, altering optimization direction.

**2. L2 Norm Clipping (Pascanu et al., 2013):**
Scales the entire gradient vector proportionally if its global Euclidean norm exceeds threshold $\tau$:
$$g_{\text{clipped}} = \begin{cases} g & \text{if } \|g\|_2 \le \tau \\ \tau \cdot \frac{g}{\|g\|_2} & \text{if } \|g\|_2 > \tau \end{cases}$$
- **Advantage:** Preserves the gradient's directional heading while strictly bounding step magnitude.

---

### 2. Scratch Python Implementation: Gradient Clipping Engine

In [ ]:
import numpy as np

def clip_by_value(gradient_vector, clip_value=1.0):
    """Clamps each element independently into [-clip_value, clip_value]."""
    return np.clip(gradient_vector, -clip_value, clip_value)

def clip_by_norm(gradient_vector, max_norm=1.0):
    """Rescales gradient vector preserving directional angle if norm > max_norm."""
    norm = np.linalg.norm(gradient_vector)
    if norm > max_norm:
        return gradient_vector * (max_norm / (norm + 1e-9)), norm
    return gradient_vector, norm

# Simulate Exploding Gradient Vector
np.random.seed(42)
g_exploding = np.array([50.0, 10.0, -80.0, 2.0])

g_val = clip_by_value(g_exploding, clip_value=5.0)
g_norm, original_norm = clip_by_norm(g_exploding, max_norm=5.0)

# Compute Cosine Similarity with Original Direction
def cosine_sim(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

sim_val = cosine_sim(g_exploding, g_val)
sim_norm = cosine_sim(g_exploding, g_norm)

print("--- Gradient Clipping Mechanics ---")
print(f"Original Gradient:    {g_exploding} | L2 Norm: {original_norm:.2f}")
print(f"Clipped by Value:     {g_val} | L2 Norm: {np.linalg.norm(g_val):.2f}")
print(f"Clipped by Norm:      {np.round(g_norm, 3)} | L2 Norm: {np.linalg.norm(g_norm):.2f}")
print(f"Direction Cosine Sim (Value): {sim_val:.4f} (Direction Distorted!)")
print(f"Direction Cosine Sim (Norm):  {sim_norm:.4f} (Direction 100% Preserved!)")

### 3. Step-by-Step Code Walkthrough
1. **Directional Invariance:** Norm clipping maintains a cosine similarity of $1.0000$ with the original gradient direction.
2. **Norm Bounding:** Caps the gradient vector norm at $\tau = 5.0$.
3. **Value Clipping Distortion:** Element-wise clamping alters the vector angle, changing the optimization descent direction.

---

### 4. Industry Applications & Trade-offs
- **LLM Pretraining (PyTorch `clip_grad_norm_`):** Guards training runs against loss spikes caused by corrupt training data.
- **Trade-off:** Clipping too aggressively can slow down optimization along flat loss surfaces.

---

### 5. Top Interview Questions & Answers
**Q1: Why is clipping by norm mathematically preferred over clipping by value?**
> **Answer:** Norm clipping rescales the gradient vector uniformly, preserving its geometric orientation. Value clipping alters relative coordinates, changing the descent direction and potentially degrading convergence.

**Q2: What is the typical clipping threshold $\tau$ used in LLM pretraining?**
> **Answer:** Foundation models typically use an L2 global norm clipping threshold of $\tau = 1.0$.